# Lecture 5 — Expectations and Transition Dynamics
## 第五讲 —— 预期与过渡动态

**Computational Methods for Heterogeneous-Agent Macro**  
**异质性主体宏观的计算方法**

Jeffrey Sun, University of Toronto, May 15, 2026

Companion to `slides/L05_transition_dynamics/L05.pdf`. Five sections,
mirroring the slides:

1. What is `HouseholdStages`?
2. Replicate the L04 Aiyagari steady state using the package.
3. The MIT-shock concept.
4. How to solve an MIT transition, in the abstract.
5. Implementation: warm start, sweeps, tatonnement, IRFs, diagnostics.

Self-contained: depends only on `HouseholdStages` (the package) and
`Plots` (figures).

与幻灯片 `slides/L05_transition_dynamics/L05.pdf` 配套，五节同序：

1. 什么是 `HouseholdStages`？
2. 用这个包复现第四讲的 Aiyagari 稳态。
3. MIT 冲击概念。
4. 抽象地讲，怎么解 MIT 过渡。
5. 实现：热启动、扫描、tatonnement、IRF、诊断。

完全自包含：只依赖 `HouseholdStages` 与 `Plots`。

### Environment
### 运行环境

In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using Plots, Printf
using LinearAlgebra
using HouseholdStages

## 1. What is `HouseholdStages`?
## 1. 什么是 `HouseholdStages`？

L04 promised "optimized implementations of Stages" you could
download and use. That package is `HouseholdStages.jl`, and we meet
it today.

What it gives you:
- **Stage primitives** — `MarkovAlong`, `WealthChange`,
  `ConsumptionSavings`, `LogitChoice`, `Migration`,
  `BorrowingConstraint`, …
- **Composition** — `∘ₛ` chains Stages in time order;
  `lift_moments` attaches integrals against the chain's terminal $\Lambda$.
- **Three inner-solve helpers** — `solve_vfi_steady_state_given_env!`,
  `solve_lambda_steady_state_given_env!`,
  `solve_steady_state_given_env!` (a bundle of the two).
- **Functorial lifts** and **SSJ utilities** (out of scope today).

What it does **not** do:
- It does not decide your aggregate state, prices, or calibration.
- It does not run the outer Walrasian tatonnement.
- It does not define transition paths.

The principle: **anything that operates only on the household chain
at a given env lives in the library; everything else is yours.**

L04 承诺过可以下载的 "优化 Stage 实现"，那就是 `HouseholdStages.jl`，今天见。

包给了你：
- **Stage 原语** —— `MarkovAlong`、`WealthChange`、`ConsumptionSavings`、
  `LogitChoice`、`Migration`、`BorrowingConstraint` 等。
- **组合** —— `∘ₛ` 按时间顺序串 Stage；`lift_moments` 把积分挂到
  链尾 $\Lambda$ 上。
- **三个内层求解器** —— `solve_vfi_steady_state_given_env!`、
  `solve_lambda_steady_state_given_env!`、`solve_steady_state_given_env!`（前两者的捆绑）。
- **函子提升**与 **SSJ 工具**（今天不讲）。

包**不**做：
- 不决定你的总量状态、价格、校准。
- 不跑外层 Walras tatonnement。
- 不定义过渡路径。

原则：**凡是在给定 env 下只在家庭链上做的事，都在库里；其余你写。**

## 2. Replicate the Aiyagari steady state using `HouseholdStages`
## 2. 用 `HouseholdStages` 复现 Aiyagari 稳态

Same model as L04: three Stages, Cobb-Douglas firm, equilibrium $\bar K = \bar K^{\mathrm{supplied}}$.
Different code path: the Stages come from the library, and we use damped **tatonnement** on $\bar K$
in the outer loop instead of L04's bisection. (Tatonnement is the only outer loop that survives the
transition case in section 5, so we adopt it here for continuity.)

与 L04 同一个模型：三个 Stage、Cobb-Douglas 厂商、均衡 $\bar K = \bar K^{\mathrm{supplied}}$。
代码路径不同：Stage 从库里来；外层用阻尼 **tatonnement** 代替 L04 的二分。
（第 5 节的过渡情形只有 tatonnement 还能用，这里先统一好。）

### 2.1 Parameters and layout / 参数与状态空间

In [ ]:
@kwdef struct AiyagariParams
    β :: Float64 = 0.96
    σ :: Float64 = 1.5
    α :: Float64 = 0.36
    δ :: Float64 = 0.08
    L :: Float64 = 1.0
    y_grid :: Vector{Float64} = [0.6, 1.0, 1.4]
    P_y    :: Matrix{Float64} = [0.7 0.2 0.1;
                                 0.2 0.6 0.2;
                                 0.1 0.2 0.7]
    N_w   :: Int     = 400
    w_min :: Float64 = 0.0
    w_max :: Float64 = 100.0
end
Base.Broadcast.broadcastable(p::AiyagariParams) = Ref(p)

# Exponentially-spaced wealth grid: dense near 0, coarse near the
# top — required by WealthChange.backward's linear V interpolation.
# 指数财富格点：0 附近密，上端稀 —— WealthChange.backward 的
# 线性 V 插值要求如此。
function exp_wealth_grid(lo, hi, n; shift = 1.0)
    return [exp(t) * shift - shift + lo
            for t in range(0.0, log((hi - lo + shift) / shift); length = n)]
end

function aiyagari_layout(p::AiyagariParams)
    return StateLayout(
        StateAxis(:wealth, continuous_grid(exp_wealth_grid(p.w_min, p.w_max, p.N_w))),
        StateAxis(:income, discrete_finite(p.y_grid)),
    )
end

p = AiyagariParams()
dims = layout_size(aiyagari_layout(p))
@printf "Calibration: β = %.3f, σ = %.2f, α = %.2f, δ = %.2f, L = %.1f\n" p.β p.σ p.α p.δ p.L
@printf "Layout: wealth %d × income %d = %d cells\n" dims[1] dims[2] prod(dims)

### 2.2 The three Stages / 三个 Stage

Each Stage corresponds exactly to one of L04's hand-rolled functions.
Same math; the package handles the V/Λ duality and the buffer management.

每个 Stage 都对应 L04 里你手写的一个函数。数学一样；V/Λ 对偶与缓冲区管理
由库代劳。

In [ ]:
_u_crra(c, ::Val{1}) = log(c)
_u_crra(c, ::Val{σ}) where σ = (c^(1 - σ)) / (1 - σ)
u_crra(c, valσ) = c < 0 ? -Inf : _u_crra(c, valσ)

function aiyagari_household(p::AiyagariParams)
    layout = aiyagari_layout(p)

    # Stage 1 — IncomeShock: Markov along the :income axis.
    # L04's `income_shock_backward(V, p) = V * Π'`, in one line.
    shock = MarkovAlong(layout; axis = :income, transition = p.P_y)

    # Stage 2 — IncomeReceipt: deterministic wealth update
    # b_post = (1+r) b + w y. L04's `income_backward` did the same,
    # snapping to nearest grid; here the library does linear V
    # interpolation backward and share-based Λ redistribution forward.
    receipt = WealthChange(layout;
        wealth_post  = (cell; env) -> (e = env[];
                        (1 + e.r) * cell.wealth + e.w * cell.income),
        wealth_axis  = :wealth,
        closure_deps = (:r, :w),
    )

    # Stage 3 — ConsumptionSavings: argmax of u(b - b_end) + β V over
    # b_end on the wealth grid. L04's `consumption_saving_backward`
    # built the N×N trial matrix and took `maximum`; here the library
    # uses monotone-policy divide-and-conquer for O(N log N).
    savings = ConsumptionSavings(layout;
        β               = p.β,
        utility         = (cell, c; env) -> u_crra(c, Val(p.σ)),
        wealth_axis     = :wealth,
        monotone_search = :divide_conquer,
    )

    # Compose the three Stages in time order, then lift the K_supplied
    # moment (∫ wealth dΛ at the chain's end).
    return lift_moments(shock ∘ₛ receipt ∘ₛ savings;
        K_supplied = at_end(integrand = :wealth, reduce = sum),
    )
end

function aiyagari_prices(K, p::AiyagariParams)
    (; α, δ, L) = p
    r = α * (K / L)^(α - 1) - δ
    w = (1 - α) * (K / L)^α
    return (; r, w)
end

hh = aiyagari_household(p)
@printf "hh: %s\n" typeof(hh).name.name

### 2.3 Inner solve at a single $\bar K$ / 单 $\bar K$ 下的内层解

Three lines:
1. Build the env: $(K, r, w)$.
2. `solve_steady_state_given_env!` iterates V backward to its fixed
   point and Λ forward to stationarity, at the given env.
3. `compute_moments` reads off $K_t^{\mathrm{supplied}}$ from the
   lifted moment.

**A heads-up about the number you're about to see.** At $\bar K = 5.0$, $r$ is high enough
that $\beta(1+r) > 1$: in the long run, individual households accumulate against the wealth
ceiling. So $K^{\mathrm{supplied}}$ at this $\bar K$ is *much* bigger than $\bar K$ — that's
the disequilibrium reality, not a bug. The outer tatonnement in 2.4 walks $\bar K$ up
gradually, crossing the impatience watershed $\beta(1+r) = 1$, and lands at equilibrium.

三行：
1. 造 env：$(K, r, w)$。
2. `solve_steady_state_given_env!` 在给定 env 下把 V 后向到不动点、Λ 前向到平稳。
3. `compute_moments` 从挂的矩里读 $K_t^{\mathrm{supplied}}$。

**关于即将看到的数字。**$\bar K = 5.0$ 处 $r$ 较高，$\beta(1+r) > 1$：长期看，单个家庭会一路
存钱到财富上限。所以这个 $\bar K$ 下的 $K^{\mathrm{supplied}}$ 比 $\bar K$ 大得多——这是
非均衡现实，不是 bug。第 2.4 节的外层 tatonnement 会逐步把 $\bar K$ 推大，跨过
$\beta(1+r) = 1$ 的不耐烦分水岭，最终落到均衡。

In [ ]:
caches, scratches = allocate(hh)
V = zeros(Float64, dims...)
Λ = fill(1.0 / prod(dims), dims...)

K_trial = 5.0
env_trial = (; K = K_trial, aiyagari_prices(K_trial, p)...)
info = solve_steady_state_given_env!(hh, env_trial, V, Λ, caches, scratches)
K_supplied_trial = compute_moments(hh, env_trial).K_supplied

@printf "K_trial = %.4f → K_supplied = %.4f (residual = %+.4f)\n" K_trial K_supplied_trial (K_supplied_trial - K_trial)
@printf "  VFI: %d iters; Λ: %d iters; r = %.4f, w = %.4f\n" info.vfi_iters info.lambda_iters env_trial.r env_trial.w

### 2.4 Outer tatonnement on $\bar K$ / 对 $\bar K$ 的外层 tatonnement

Hand-rolled — "close-the-model" outer loops are the consumer's responsibility
in this library. Damped: $\bar K \leftarrow \bar K + \mathrm{lr}\,(\bar K^{\mathrm{supplied}} - \bar K)$.
Stop when relative error is below `rtol`.

外层手写——"闭合模型"的循环由调用方负责。阻尼形式：
$\bar K \leftarrow \bar K + \mathrm{lr}\,(\bar K^{\mathrm{supplied}} - \bar K)$。
相对误差降到 `rtol` 之下时停。

In [ ]:
function aiyagari_steady_state(p::AiyagariParams;
                                K_init       = 5.0,
                                update_speed = 0.05,
                                rtol         = 2e-2,
                                max_iter     = 500)
    hh   = aiyagari_household(p)
    caches, scratches = allocate(hh)
    dims = layout_size(aiyagari_layout(p))
    V    = zeros(Float64, dims...)
    Λ    = fill(1.0 / prod(dims), dims...)

    K = K_init
    iters = 0
    K_err = Inf
    residual_history = Float64[]
    while iters < max_iter
        env  = (; K, aiyagari_prices(K, p)...)
        info = solve_steady_state_given_env!(hh, env, V, Λ, caches, scratches)
        V, Λ = info.V, info.Λ
        K_supplied = compute_moments(hh, env).K_supplied
        K_err = abs(K_supplied - K) / K
        push!(residual_history, K_err)
        iters += 1
        K_err <= rtol && break
        K += update_speed * (K_supplied - K)
    end
    converged = K_err <= rtol
    (; r, w) = aiyagari_prices(K, p)
    return (; K, r, w, V, Λ, iters, converged, residual_history)
end

ss = aiyagari_steady_state(p)
@printf "K_ss = %.4f, r_ss = %.4f, w_ss = %.4f; converged = %s in %d iters\n" ss.K ss.r ss.w ss.converged ss.iters

**Comparison to L04.** Same model, same equilibrium concept. The L04 solver landed at $\bar K \approx 5.30$
with a coarser hand-rolled grid and bisection; here the 400-point exponential grid and a tighter
tatonnement stop give $\bar K \approx 5.68$. The economics is the same; the precision is sharper.

**与 L04 对比。**模型一样，均衡概念一样。L04 用较粗的手写格点 + 二分，得 $\bar K \approx 5.30$；
这里 400 点指数格点 + 更紧的 tatonnement 停止，得 $\bar K \approx 5.68$。经济学一样，精度更高。

## 3. The MIT shock concept
## 3. MIT 冲击概念

A **steady state** is a single $(K, V, \Lambda)$. A **transition path** is a sequence
$\{(K_t, V_t, \Lambda_t)\}_{t=1,\dots,T}$.

An **MIT shock** is:
- at $t = 0$, the economy is at steady state;
- at $t = 1$, a one-time impulse hits — agents are *surprised*;
- from $t = 1$ onward, agents have **perfect foresight** of the entire future path of the shock.

Surprise + perfect foresight is unrealistic but tractable: no expectations integrals,
no aggregate randomness. The simplest deterministic dynamic experiment beyond steady state.

**稳态**是单点 $(K, V, \Lambda)$。**过渡路径**是序列 $\{(K_t, V_t, \Lambda_t)\}_{t=1,\dots,T}$。

**MIT 冲击**：
- $t = 0$ 时经济处于稳态；
- $t = 1$ 时一次性冲击来袭，主体*毫无预期*；
- $t = 1$ 起，主体**完美预期**冲击的全部未来路径。

惊喜 + 完美预期不现实但可解：没有期望积分、没有总量随机性。是超越稳态的最简单确定性动态实验。

### 3.1 Our example — TFP impulse / 我们的示例：TFP 冲击

A one-time positive TFP innovation at $t = 1$, then AR(1) decay back to baseline:

$$
  A_1 = 1.05, \qquad A_{t+1} = 1 + \rho\,(A_t - 1), \quad \rho = 0.85.
$$

$\{A_t\}$ is the **only** exogenous driver. The path of $K$, of prices, of $V$, of $\Lambda$ — all the response.

$t = 1$ 一次正向 TFP 冲击，之后按 AR(1) 衰回基线：上式。$\{A_t\}$ 是**唯一**外生驱动，其余皆响应。

In [ ]:
# tfp_path is defined locally — the library leaves transition
# utilities to the consumer.
# tfp_path 本地定义 —— 库把过渡相关的工具交给调用方。
function tfp_path(T; A_0 = 1.05, ρ = 0.85, A_ss = 1.0)
    A = zeros(T); A[1] = A_0
    for t in 2:T
        A[t] = A_ss + ρ * (A[t-1] - A_ss)
    end
    return A
end

T = 100
A_path = tfp_path(T; A_0 = 1.05, ρ = 0.85, A_ss = 1.0)

plot(1:T, A_path; lw = 2, xlabel = "period t", ylabel = "A_t",
     label = "AR(1) TFP path  (A_0 = 1.05, ρ = 0.85)",
     title = "Exogenous TFP path", legend = :topright)
hline!([1.0]; color = :gray, linestyle = :dash, label = "A_ss = 1")

**A new TFP price function.** With time-varying $A_t$, the price function has to take $A$ explicitly:

**新的含 TFP 价格函数。**$A_t$ 时变，价格函数得显式接 $A$：

In [ ]:
function mit_prices(K, A, p::AiyagariParams)
    (; α, δ, L) = p
    r = α * A * (K / L)^(α - 1) - δ
    w = (1 - α) * A * (K / L)^α
    return (; r, w)
end

# Sanity check: at A = 1 we recover aiyagari_prices.
@printf "mit_prices(K=5.0, A=1.0) = %s\n" mit_prices(5.0, 1.0, p)
@printf "aiyagari_prices(K=5.0)   = %s\n" aiyagari_prices(5.0, p)

## 4. Solving an MIT transition — the abstract algorithm
## 4. 解 MIT 过渡 —— 抽象算法

**Given.** The exogenous path $\{A_t\}_{t=1}^T$, the household chain `hh`, and the firm $(\alpha, \delta, L)$.

**Find.** Sequences $\{K_t\}, \{V_t\}, \{\Lambda_t\}$ such that
- $V_{T+1} = V_{\mathrm{ss}}$ (terminal condition),
- $\Lambda_1 = \Lambda_{\mathrm{ss}}$ (initial condition),
- $V_t = \texttt{backward!}(V_{t+1}, \mathrm{env}_t)$,
- $\Lambda_{t+1} = \texttt{forward!}(\Lambda_t)$,
- $K_t = \int b\,\mathrm{d}\Lambda_t = K_t^{\mathrm{supplied}}$ (market clears every period).

**Three ingredients.**

1. **Backward sweep on $V$.** $V_T \leftarrow V_{\mathrm{ss}}$, then walk backward: value flows from future to present (Bellman). $T$ calls to `backward!`.
2. **Forward sweep on $\Lambda$.** $\Lambda_1 \leftarrow \Lambda_{\mathrm{ss}}$, then walk forward: mass flows in time. $T$ calls to `forward!`.
3. **Outer tatonnement on $\{K_t\}$.** Damped Walrasian update: $K_t \leftarrow (1-d) K_t + d K_t^{\mathrm{supplied}}$.

**Why no bisection?** The unknown $\{K_t\}$ is $T$-dimensional — there is no scalar to bracket.
Damped tatonnement is the simplest method that works in $T$-dim. Newton with the sequence-space
Jacobian (next lecture) is faster but the loop structure is the same.

**已知。**外生 $\{A_t\}_{t=1}^T$、家庭链 `hh`、厂商 $(\alpha, \delta, L)$。

**求。**序列 $\{K_t\}, \{V_t\}, \{\Lambda_t\}$ 满足以上五条。

**三个要件：**
1. **$V$ 后向扫。**$V_T \leftarrow V_{\mathrm{ss}}$，向前倒走：价值从未来传到现在（Bellman）。调 $T$ 次 `backward!`。
2. **$\Lambda$ 前向扫。**$\Lambda_1 \leftarrow \Lambda_{\mathrm{ss}}$，向后推：质量沿时间前流。调 $T$ 次 `forward!`。
3. **对 $\{K_t\}$ 做外层 tatonnement。**$K_t \leftarrow (1-d) K_t + d K_t^{\mathrm{supplied}}$。

**为什么不能二分？**未知 $\{K_t\}$ 是 $T$ 维的，没有标量可二分。$T$ 维下阻尼 tatonnement
是最简能用的方法；下讲用序列空间 Jacobian 的 Newton 更快，但循环结构一样。

## 5. Implementation
## 5. 实现

Five subsections matching the slides:
1. Warm start from the section-2 SS.
2. Backward sweep on $V$.
3. Forward sweep on $\Lambda$.
4. Outer tatonnement update.
5. Run, IRFs, diagnostics.

We wrap the whole transition into one function `mit_shock_transition` so we can call it
with different damping values in the diagnostics subsection.

五个小节，对应幻灯片：热启动、$V$ 后向、$\Lambda$ 前向、外层 tatonnement、运行 + IRF + 诊断。
整段封进 `mit_shock_transition` 函数，方便诊断小节用不同阻尼跑。

In [ ]:
function mit_shock_transition(p::AiyagariParams;
                               T::Int       = 100,
                               A_0::Float64 = 1.05,
                               ρ::Float64   = 0.85,
                               damping      = 0.2,
                               tol          = 1e-3,
                               max_iter     = 200)
    # 5.1 Warm start: section-2 steady state at A = 1.
    ss = aiyagari_steady_state(p)
    K_ss, V_ss, Λ_ss = ss.K, ss.V, ss.Λ

    # Fresh chain + workspace for the path solve.
    hh = aiyagari_household(p)
    caches, scratches = allocate(hh)
    dims = layout_size(aiyagari_layout(p))

    # V_path[T+1] = V_ss; V_path[1..T] filled by backward sweep.
    # Λ_path[1]   = Λ_ss; Λ_path[2..T+1] filled by forward sweep.
    V_path = [copy(V_ss) for _ in 1:(T + 1)]
    Λ_path = [zeros(Float64, dims...) for _ in 1:(T + 1)]
    Λ_path[1] .= Λ_ss

    # Exogenous TFP path; initial K-path guess = constant SS.
    A_path = tfp_path(T; A_0 = A_0, ρ = ρ, A_ss = 1.0)
    K_t        = fill(K_ss, T)
    K_supplied = zeros(T)
    residual_history = Float64[]
    iters = 0
    res = Inf

    while iters < max_iter
        # 5.2 Backward sweep: V_{T+1} = V_ss → V_T → … → V_1.
        for t in T:-1:1
            env = (; K = K_t[t], mit_prices(K_t[t], A_path[t], p)...)
            V_path[t] .= backward!(hh, V_path[t + 1], env, caches, scratches)
        end

        # 5.3 Forward sweep: re-seat caches at each period-t env, then
        # push Λ and read off K_supplied.
        for t in 1:T
            env = (; K = K_t[t], mit_prices(K_t[t], A_path[t], p)...)
            backward!(hh, V_path[t + 1], env, caches, scratches)   # re-seat policy
            Λ_path[t + 1] .= forward!(hh, Λ_path[t], caches, scratches)
            K_supplied[t]   = compute_moments(hh, env).K_supplied
        end

        # 5.4 Outer tatonnement update.
        res = maximum(abs, K_supplied .- K_t)
        push!(residual_history, res)
        iters += 1
        res <= tol && break
        K_t .= (1 - damping) .* K_t .+ damping .* K_supplied
    end

    converged = res <= tol
    return (; K_path = K_t, A_path, V_path, Λ_path,
              K_ss, V_ss, Λ_ss,
              iters, converged, residual_history)
end

### 5.5 Run it / 跑一下

In [ ]:
tr = mit_shock_transition(p; T = T, A_0 = 1.05, ρ = 0.85, damping = 0.2, tol = 1e-3, max_iter = 200)
@printf "converged = %s in %d outer iterations\n" tr.converged tr.iters
@printf "K_ss            = %.4f\n" tr.K_ss
@printf "K[1]   (impact) = %.4f  (Δ = %+0.4f)\n" tr.K_path[1] (tr.K_path[1] - tr.K_ss)
@printf "K[5]            = %.4f  (Δ = %+0.4f)\n" tr.K_path[5] (tr.K_path[5] - tr.K_ss)
@printf "K[20]           = %.4f  (Δ = %+0.4f)\n" tr.K_path[20] (tr.K_path[20] - tr.K_ss)
@printf "K[100] (≈end)   = %.4f  (Δ = %+0.4f)\n" tr.K_path[100] (tr.K_path[100] - tr.K_ss)

### 5.6 Impulse responses / 脉冲响应

Capital first; then $r$ and $w$.

先看资本，再看 $r$ 与 $w$。

In [ ]:
plot(1:T, tr.K_path; lw = 2, label = "K_t (transition)",
     xlabel = "period t", ylabel = "aggregate capital",
     title = "IRF: K to a +5% TFP shock with ρ = 0.85")
hline!([tr.K_ss]; color = :gray, linestyle = :dash, label = "K_ss")

In [ ]:
r_path = [mit_prices(tr.K_path[t], tr.A_path[t], p).r for t in 1:T]
w_path = [mit_prices(tr.K_path[t], tr.A_path[t], p).w for t in 1:T]

plot(layout = (2, 1), size = (700, 500))
plot!(1:T, r_path; subplot = 1, lw = 2, label = "r_t",
      xlabel = "period", ylabel = "r", title = "IRF: real rate")
hline!([mit_prices(tr.K_ss, 1.0, p).r];
       subplot = 1, color = :gray, linestyle = :dash, label = "r_ss")
plot!(1:T, w_path; subplot = 2, lw = 2, label = "w_t",
      xlabel = "period", ylabel = "w", title = "IRF: wage")
hline!([mit_prices(tr.K_ss, 1.0, p).w];
       subplot = 2, color = :gray, linestyle = :dash, label = "w_ss")

### 5.7 Residual history / 残差历史

Damped tatonnement drops the residual geometrically until it hits a *discretization floor* at
$\sim 2.5 \times 10^{-3}$. The floor is the hard-`argmax` `ConsumptionSavings` policy flipping
between adjacent grid cells as $K_t$ wobbles. A smoothed (`LogitChoice`-based) savings policy
or a tighter wealth grid would push the floor down.

阻尼 tatonnement 残差几何下降，直到撞上 $\sim 2.5 \times 10^{-3}$ 的*离散化下限*。
下限来自硬 `argmax` 的 `ConsumptionSavings`：$K_t$ 微动时策略在相邻格子间翻转。
把储蓄换成平滑（`LogitChoice`）或加密财富格点，下限就会下降。

In [ ]:
plot(1:length(tr.residual_history), tr.residual_history;
     yscale = :log10, lw = 2, marker = :circle, markersize = 3,
     xlabel = "outer iteration", ylabel = "‖K^supplied − K‖∞",
     label = "residual", title = "Damped tatonnement residual history")

### 5.8 Try this — damping sweep / 动手试 —— 阻尼扫一遍

Damping is a craft: too high oscillates, too low crawls. Re-run the transition at
$d \in \{0.1, 0.2, 0.4, 0.6\}$ and overlay the residual histories.

阻尼是个手艺活：太大振荡，太小爬不动。把过渡在 $d \in \{0.1, 0.2, 0.4, 0.6\}$
重新跑几遍，把残差历史叠在一张图上看看。

**Expected:** $d \in \{0.4, 0.6\}$ fails to converge (residual oscillates or grows);
$d = 0.1$ converges with a shallow slope; $d = 0.2$ is roughly the sweet spot.

**预期：**$d \in \{0.4, 0.6\}$ 不收敛（残差振荡或增大）；$d = 0.1$ 缓慢线性下降；
$d = 0.2$ 大约是最佳位置。

In [ ]:
plt = plot(yscale = :log10, xlabel = "outer iteration",
           ylabel = "residual ‖K^supplied − K‖∞",
           title  = "Damping sweep")
for d in (0.1, 0.2, 0.4, 0.6)
    res = mit_shock_transition(p; T = T, A_0 = 1.05, ρ = 0.85,
                                 damping = d, tol = 1e-3, max_iter = 80)
    plot!(plt, 1:length(res.residual_history), res.residual_history;
          lw = 2, label = "d = $(d)")
end
plt

## Where to next
## 接下来

**L6 (Krusell–Smith)** adds aggregate uncertainty: $A_t$ becomes a stochastic process and the
equilibrium object becomes a function of the aggregate state, not a single path.

**L7 (CV is all you need)** replaces L6's inner loop with a neural net trained on continuation
values — the climax of the course.

**Also in the library** (out of scope today): `expectation_vectors`, `build_F`, `J_from_F` —
the sequence-space Jacobian via the fake-news algorithm. See
`HouseholdStages/examples/notebooks/aiyagari_mit_shock.jl` (section 5) for an end-to-end demo.

**第六讲（Krusell–Smith）**加入总量不确定性：$A_t$ 变成随机过程，均衡对象变成总量状态的函数，
而非单一路径。

**第七讲（续值即一切）**用神经网络替换第六讲的内层 —— 课程高潮。

**库里还有**（今天不讲）：`expectation_vectors`、`build_F`、`J_from_F` —— fake-news 算法的
序列空间 Jacobian。端到端演示见 `HouseholdStages/examples/notebooks/aiyagari_mit_shock.jl`（第 5 节）。